In [ ]:

import time
import pandas as pd
from datetime import datetime, timedelta
import sys
sys.path.insert(0, r"D:\ETUDEµ\CITE\ETAPE4\PROJET DE FIN DE SESSION\TOURMANT 1")

from system_automatic.hse_analysis_system_save_bd import HSEAnalysisSystem



# 1. Initialiser le système une seule fois
hse_system = HSEAnalysisSystem()

# 2. Exécuter en mode oneshot
#df_result = hse_system.run(mode='oneshot', lookback_minutes=2)

# 3. Ou lancer en temps réel (cycle)
hse_system.run(mode='cycle', lookback_minutes=1, frequency_seconds=10,camera_id=11)





In [ ]:
from system_automatic.hse_analysis_system_save_bd import HSEAnalysisSystem

# 1. Initialiser le système une seule fois
hse_system = HSEAnalysisSystem()

# 2. Exécuter en mode oneshot
#df_result = hse_system.run(mode='oneshot', lookback_minutes=2)

# 3. Ou lancer en temps réel (cycle)
# Modifiez la dernière ligne de votre script de lancement :
hse_system.run(mode='cycle', lookback_minutes=5, frequency_seconds=5, camera_id=11)

Ce code permet de faire l'interférence

Après vous pouvez utiliser l'autre pour la visualisation

In [2]:
import sys
import threading
import time

# S'assurer que le chemin est correct
sys.path.insert(0, r"C:\TOURMANT 1")

from db.pool import init_pools
from inference.hse_acquisition_manager import HSEAcquisitionManager
from system_automatic.hse_analysis_system_save_bd import HSEAnalysisSystem

# --- CONFIGURATION ---
SITE_ID = 2
LOOKBACK = 1        # minutes d'historique à analyser
FREQUENCY = 10      # secondes entre chaque scan du site

def main():
    # 1. INITIALISATION DES POOLS
    init_pools(read_size=10, write_size=10)

    # 2. DÉMARRAGE DE L'ACQUISITION (Flux Vidéo + YOLO -> DB)
    # Ce manager gère ses propres threads internes
    print("Démarrage de l'acquisition vidéo...")
    acquisition_manager = HSEAcquisitionManager(site_id=SITE_ID)
    acquisition_manager.start_dual_flux_pipeline()

    # Laisser quelques secondes pour que les premières détections arrivent en DB
    time.sleep(5)

    # 3. DÉMARRAGE DE L'ANALYSE HSE (DB -> Pipeline Risque -> DB Alerts)
    print("Démarrage du pipeline d'analyse des risques...")
    hse_system = HSEAnalysisSystem()
    
    # Lancement du mode automatique par site
    hse_system.run(
        site_id=SITE_ID, 
        lookback_minutes=LOOKBACK, 
        frequency_seconds=FREQUENCY
    )

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
if __name__ == "__main__":
    main()

Démarrage de l'acquisition vidéo...
 [OK] Source Webcam pour : CAM01
 Vidéo démarrée : data\recordings\cam_11\20260409_170236.avi (640x480)
Démarrage du pipeline d'analyse des risques...
Chargement du modèle ML : C:\TOURMANT 1\dataset machine Learning\model_hse_v1.pkl...
 IA Cam 11: 6 objets détectés
 IA Cam 11: 5 objets détectés
Chargement des règles : C:\TOURMANT 1\system_automatic\hse_rules.json
--- Démarrage Système HSE Global (Site 2) ---

========== ACTIVITY PREDICTION ==========
Features utilisées par le modèle :
['avg_num_persons' 'max_num_persons' 'unique_person_tracks' 'person_density' 'person_presence_ratio' 'avg_epi_compliance' 'helmet_compliance_ratio' 'vest_compliance_ratio' 'num_no_helmet_events' 'num_no_vest_events' 'num_machines_avg' 'machine_presence_ratio' 'unique_machine_tracks' 'co_presence_person_machine'
 'dangerous_proximity_frames' 'min_person_machine_distance' 'avg_person_speed' 'std_person_speed' 'sudden_stop_events' 'erratic_motion_score' 'direction_variance

In [ ]:
pip install mysql-connector-python